# 🎯 QWEN2-VL-2B QLORA TRAINING & BENCHMARKING ON TESLA T4 GPU
Huấn luyện căn chỉnh định dạng (Format Alignment) và tăng cường năng lực trích xuất hóa đơn tiếng Việt.

In [ ]:
# 1. Cài đặt thư viện môi trường
!pip uninstall -y -q torchao
!pip install -q --no-deps qwen-vl-utils==0.0.8
!pip install -q "transformers==4.46.2" "peft==0.13.2" "accelerate==0.34.2" bitsandbytes pillow torchvision pyyaml

import sys
for mod in list(sys.modules.keys()):
    if any(mod.startswith(k) for k in ["transformers", "peft", "accelerate", "torchao", "qwen_vl_utils"]):
        del sys.modules[mod]

import os
import gc
import time
import json
import re
import zipfile
from pathlib import Path
import torch
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from qwen_vl_utils import process_vision_info

print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
print(f"🧠 Tổng VRAM khả dụng: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. Giải nén dữ liệu hình ảnh
extract_dir = "/kaggle/working/images"
os.makedirs(extract_dir, exist_ok=True)

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".zip"):
            print(f"📦 Đang giải nén: {f}")
            try:
                with zipfile.ZipFile(os.path.join(root, f), 'r') as zf:
                    zf.extractall(extract_dir)
            except Exception as e:
                print(f"   Lỗi giải nén {f}: {e}")

image_map = {}
for root, dirs, files in os.walk("/kaggle"):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            image_map[f] = os.path.join(root, f)

print(f"📸 Tổng số ảnh đã lập chỉ mục trên Kaggle: {len(image_map)}")


In [ ]:
# 3. Chuẩn bị Dataset & Custom Data Collator
model_id = "Qwen/Qwen2-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=768*28*28)

raw_train_samples = [
  {
    "image_name": "cafe_highlands_train_001.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_001.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_001.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_001.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_001.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_001.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_002.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_002.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_002.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_002.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_002.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_002.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_003.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_003.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_003.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_003.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_003.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_003.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_005.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_005.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_005.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_005.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_005.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_005.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_006.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_006.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_006.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_006.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_006.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_006.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_007.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_007.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_007.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_007.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_007.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_007.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_008.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_008.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_008.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_008.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_008.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_008.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_009.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_009.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_009.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_009.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_009.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_009.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_010.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_010.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_010.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_010.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_010.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_010.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_011.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_011.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_011.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_011.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_011.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_011.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_012.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_012.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_012.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_012.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_012.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_012.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_013.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_013.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_013.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_013.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_013.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_013.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_014.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_014.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_014.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_014.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_014.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_014.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_015.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_015.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_015.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_015.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_015.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_015.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_016.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_016.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_016.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_016.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_016.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_016.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_017.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_017.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_017.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_017.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_017.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_017.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_018.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_018.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_018.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_018.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_018.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_018.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_019.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_019.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_019.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_019.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_019.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_019.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_020.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_020.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_020.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_020.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_020.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_020.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_021.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_021.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_021.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_021.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_021.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_021.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_022.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_022.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_022.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_022.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_022.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_022.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_023.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_023.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_023.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_023.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_023.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_023.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_024.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_024.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_024.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_024.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_024.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_024.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_026.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_026.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_026.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_026.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_026.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_026.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_027.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_027.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_027.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_027.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_027.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_027.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_028.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_028.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_028.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_028.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_028.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_028.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_029.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_029.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_029.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_029.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_029.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_029.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_030.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_030.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_030.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_030.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_030.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_030.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_031.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_031.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_031.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_031.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_031.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_031.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_032.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_032.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_032.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_032.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_032.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_032.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_033.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_033.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_033.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_033.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_033.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_033.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_034.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_034.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_034.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_034.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_034.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_034.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_035.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_035.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_035.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_035.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_035.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_035.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_036.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_036.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_036.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_036.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_036.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_036.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_037.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_037.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_037.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_037.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_037.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_037.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_038.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_038.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_038.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_038.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_038.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_038.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_039.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_039.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_039.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_039.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_039.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_039.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_040.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_040.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_040.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_040.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_040.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_040.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_041.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_041.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_041.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_041.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_041.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_041.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_042.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_042.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_042.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_042.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_042.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_042.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_043.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_043.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_043.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_043.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_043.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_043.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_044.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_044.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_044.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_044.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_044.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_044.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_045.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_045.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_045.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_045.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_045.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_045.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_046.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_046.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_046.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_046.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_046.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_046.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_047.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_047.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_047.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_047.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_047.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_047.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_048.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_048.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_048.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_048.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_048.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_048.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_049.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_049.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_049.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_049.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_049.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_049.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_050.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_050.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_050.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_050.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_050.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_050.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_051.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_051.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_051.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_051.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_051.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_051.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_052.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_052.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_052.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_052.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_052.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_052.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_053.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_053.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_053.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_053.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_053.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_053.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_054.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_054.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_054.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_054.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_054.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_054.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_055.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_055.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_055.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_055.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_055.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_055.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_056.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_056.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_056.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_056.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_056.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_056.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_057.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_057.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_057.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_057.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_057.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_057.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_058.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_058.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_058.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_058.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_058.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_058.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_059.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_059.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_059.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_059.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_059.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_059.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_060.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_060.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_060.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_060.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_060.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_060.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_061.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_061.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_061.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_061.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_061.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_061.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_062.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_062.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_062.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_062.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_062.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_062.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_064.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_064.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_064.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_064.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_064.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_064.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_065.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_065.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_065.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_065.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_065.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_065.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_066.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_066.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_066.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_066.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_066.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_066.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_067.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_067.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_067.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_067.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_067.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_067.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_068.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_068.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_068.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_068.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_068.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_068.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_069.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_069.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_069.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_069.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_069.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_069.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_070.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_070.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_070.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_070.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_070.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_070.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_071.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_071.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_071.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_071.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_071.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_071.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_072.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_072.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_072.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_072.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_072.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_072.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_073.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_073.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_073.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_073.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_073.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_073.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_074.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_074.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_074.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_074.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_074.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_074.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_075.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_075.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_075.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_075.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_075.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_075.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_076.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_076.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_076.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_076.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_076.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_076.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_077.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_077.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_077.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_077.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_077.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_077.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_078.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_078.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_078.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_078.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_078.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_078.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_079.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_079.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_079.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_079.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_079.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_079.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_080.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_080.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_080.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_080.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_080.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_080.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_081.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_081.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_081.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_081.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_081.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_081.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_082.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_082.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_082.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_082.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_082.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_082.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_083.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_083.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_083.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_083.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_083.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_083.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_084.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_084.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_084.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_084.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_084.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_084.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_085.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_085.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_085.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_085.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_085.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_085.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_086.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_086.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_086.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_086.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_086.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_086.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_087.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_087.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_087.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_087.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_087.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_087.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_088.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_088.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_088.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_088.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_088.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_088.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_089.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_089.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_089.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_089.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_089.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_089.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_090.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_090.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_090.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_090.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_090.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_090.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_091.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_091.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_091.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_091.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_091.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_091.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_092.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_092.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_092.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_092.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_092.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_092.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_093.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_093.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_093.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_093.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_093.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_093.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_094.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_094.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_094.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_094.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_094.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_094.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_095.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_095.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_095.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_095.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_095.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_095.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_096.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_096.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_096.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_096.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_096.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_096.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_097.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_097.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_097.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_097.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_097.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_097.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_098.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_098.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_098.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_098.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_098.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_098.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_099.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_099.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_099.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_099.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_099.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_099.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_100.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_100.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_100.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_100.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_100.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_100.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_101.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_101.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_101.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_101.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_101.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_101.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_102.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_102.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_102.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_102.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_102.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_102.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_103.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_103.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_103.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_103.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_103.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_103.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_104.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_104.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_104.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_104.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_104.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_104.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_105.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_105.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_105.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_105.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_105.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_105.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_106.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_106.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_106.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_106.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_106.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_106.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_107.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_107.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_107.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_107.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_107.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_107.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_108.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_108.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_108.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_108.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_108.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_108.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_109.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_109.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_109.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_109.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_109.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_109.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_110.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_110.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_110.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_110.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_110.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_110.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_111.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_111.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_111.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_111.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_111.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_111.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_112.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_112.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_112.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_112.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_112.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_112.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_113.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_113.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_113.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_113.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_113.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_113.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_114.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_114.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_114.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_114.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_114.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_114.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_115.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_115.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_115.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_115.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_115.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_115.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_116.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_116.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_116.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_116.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_116.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_116.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_117.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_117.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_117.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_117.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_117.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_117.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_118.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_118.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_118.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_118.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_118.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_118.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_119.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_119.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_119.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_119.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_119.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_119.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_120.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_120.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_120.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_120.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_120.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_120.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_121.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_121.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_121.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_121.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_121.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_121.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_122.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_122.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_122.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_122.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_122.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_122.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_123.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_123.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_123.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_123.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_123.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_123.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_124.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_124.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_124.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_124.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_124.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_124.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_125.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_125.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_125.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_125.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_125.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_125.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_126.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_126.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_126.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_126.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_126.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_126.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_127.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_127.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_127.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_127.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_127.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_127.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_128.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_128.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_128.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_128.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_128.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_128.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_129.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_129.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_129.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_129.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_129.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_129.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_130.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_130.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_130.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_130.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_130.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_130.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_131.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_131.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_131.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_131.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_131.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_131.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_132.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_132.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_132.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_132.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_132.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_132.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_133.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_133.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_133.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_133.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_133.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_133.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_134.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_134.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_134.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_134.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_134.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_134.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_135.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_135.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_135.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_135.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_135.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_135.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_136.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_136.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_136.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_136.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_136.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_136.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_137.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_137.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_137.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_137.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_137.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_137.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_138.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_138.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_138.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_138.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_138.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_138.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_139.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_139.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_139.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_139.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_139.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_139.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_140.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_140.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_140.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_140.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_140.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_140.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_141.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_141.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_141.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_141.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_141.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_141.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_142.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_142.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_142.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_142.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_142.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_142.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_143.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_143.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_143.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_143.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_143.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_143.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_144.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_144.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_144.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_144.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_144.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_144.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_145.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_145.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_145.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_145.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_145.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_145.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_146.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_146.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_146.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_146.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_146.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_146.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_147.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_147.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_147.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_147.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_147.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_147.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_148.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_148.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_148.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_148.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_148.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_148.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_149.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_149.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_149.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_149.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_149.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_149.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_150.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_150.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_150.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_150.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_150.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_150.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_151.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_151.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_151.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_151.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_151.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_151.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_152.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_152.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_152.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_152.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_152.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_152.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_153.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_153.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_153.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_153.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_153.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_153.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_154.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_154.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_154.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_154.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_154.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_154.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_155.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_155.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_155.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_155.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_155.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_155.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_156.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_156.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_156.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_156.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_156.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_156.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_157.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_157.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_157.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_157.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_157.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_157.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_158.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_158.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_158.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_158.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_158.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_158.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_159.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_159.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_159.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_159.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_159.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_159.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_160.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_160.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_160.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_160.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_160.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_160.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_161.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_161.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_161.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_161.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_161.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_161.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_162.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_162.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_162.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_162.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_162.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_162.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_163.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_163.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_163.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_163.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_163.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_163.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_164.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_164.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_164.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_164.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_164.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_164.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_165.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_165.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_165.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_165.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_165.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_165.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_166.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_166.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_166.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_166.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_166.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_166.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_167.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_167.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_167.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_167.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_167.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_167.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_168.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_168.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_168.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_168.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_168.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_168.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_169.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_169.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_169.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_169.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_169.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_169.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_170.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_170.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_170.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_170.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_170.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_170.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_171.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_171.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_171.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_171.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_171.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_171.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_172.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_172.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_172.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_172.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_172.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_172.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_173.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_173.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_173.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_173.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_173.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_173.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_174.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_174.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_174.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_174.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_174.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_174.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_175.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_175.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_175.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_175.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_175.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_175.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_176.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_176.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_176.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_176.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_176.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_176.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_177.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_177.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_177.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_177.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_177.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_177.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_178.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_178.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_178.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_178.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_178.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_178.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_179.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_179.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_179.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_179.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_179.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_179.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_180.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_180.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_180.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_180.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_180.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_180.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_181.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_181.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_181.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_181.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_181.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_181.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_182.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_182.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_182.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_182.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_182.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_182.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_183.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_183.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_183.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_183.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_183.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_183.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_184.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_184.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_184.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_184.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_184.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_184.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_185.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_185.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_185.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_185.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_185.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_185.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_186.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_186.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_186.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_186.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_186.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_186.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_187.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_187.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_187.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_187.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_187.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_187.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_188.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_188.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_188.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_188.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_188.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_188.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_189.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_189.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_189.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_189.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_189.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_189.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_190.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_190.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_190.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_190.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_190.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_190.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_191.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_191.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_191.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_191.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_191.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_191.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_192.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_192.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_192.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_192.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_192.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_192.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_193.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_193.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_193.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_193.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_193.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_193.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_194.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_194.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_194.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_194.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_194.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_194.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_195.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_195.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_195.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_195.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_195.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_195.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_196.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_196.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_196.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_196.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_196.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_196.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_197.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_197.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_197.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_197.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_197.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_197.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_198.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_198.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_198.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_198.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_198.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_198.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_199.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_199.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_199.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_199.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_199.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_199.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_200.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_200.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_200.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_200.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_200.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_200.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_201.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_201.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_201.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_201.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_201.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_201.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_202.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_202.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_202.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_202.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_202.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_202.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_203.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_203.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_203.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_203.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_203.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_203.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_204.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_204.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_204.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_204.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_204.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_204.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_205.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_205.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_205.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_205.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_205.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_205.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_206.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_206.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_206.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_206.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_206.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_206.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_207.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_207.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_207.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_207.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_207.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_207.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_208.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_208.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_208.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_208.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_208.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_208.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_209.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_209.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_209.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_209.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_209.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_209.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_210.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_210.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_210.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_210.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_210.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_210.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_212.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_212.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_212.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_212.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_212.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_212.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_213.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_213.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_213.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_213.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_213.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_213.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_214.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_214.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_214.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_214.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_214.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_214.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_215.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_215.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_215.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_215.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_215.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_215.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_216.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_216.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_216.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_216.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_216.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_216.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_217.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_217.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_217.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_217.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_217.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_217.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_218.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_218.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_218.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_218.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_218.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_218.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_219.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_219.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_219.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_219.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_219.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_219.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_220.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_220.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_220.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_220.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_220.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_220.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_221.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_221.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_221.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_221.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_221.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_221.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_222.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_222.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_222.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_222.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_222.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_222.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_223.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_223.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_223.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_223.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_223.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_223.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_224.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_224.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_224.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_224.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_224.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_224.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_225.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_225.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_225.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_225.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_225.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_225.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_226.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_226.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_226.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_226.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_226.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_226.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_227.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_227.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_227.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_227.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_227.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_227.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_228.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_228.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_228.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_228.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_228.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_228.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_229.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_229.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_229.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_229.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_229.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_229.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_230.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_230.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_230.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_230.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_230.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_230.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_231.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_231.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_231.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_231.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_231.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_231.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_232.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_232.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_232.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_232.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_232.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_232.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_234.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_234.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_234.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_234.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_234.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_234.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_235.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_235.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_235.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_235.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_235.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_235.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_236.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_236.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_236.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_236.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_236.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_236.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_237.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_237.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_237.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_237.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_237.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_237.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_238.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_238.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_238.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_238.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_238.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_238.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_239.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_239.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_239.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_239.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_239.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_239.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_240.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_240.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_240.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_240.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_240.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_240.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_241.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_241.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_241.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_241.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_241.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_241.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_242.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_242.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_242.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_242.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_242.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_242.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_243.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_243.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_243.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_243.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_243.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_243.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_244.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_244.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_244.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_244.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_244.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_244.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_245.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_245.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_245.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_245.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_245.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_245.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_246.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_246.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_246.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_246.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_246.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_246.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_247.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_247.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_247.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_247.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_247.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_247.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_248.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_248.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_248.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_248.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_248.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_248.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_249.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_249.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_249.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_249.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_249.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_249.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_250.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_250.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_250.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_250.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_250.png",
    "messages": []
  },
  {
    "image_name": "cafe_highlands_train_250.png",
    "messages": []
  }
]

# Chuẩn hóa đường dẫn ảnh trong messages
valid_train_data = []
for item in raw_train_samples:
    img_name = item["image_name"]
    if img_name in image_map:
        real_img_path = image_map[img_name]
        msgs = item["messages"]
        # Thay đổi image path thành PIL Image hoặc đường dẫn thật
        new_msgs = []
        for m in msgs:
            role = m["role"]
            content = m["content"]
            if isinstance(content, list):
                new_content = []
                for c in content:
                    if c.get("type") == "image":
                        new_content.append({"type": "image", "image": real_img_path})
                    else:
                        new_content.append(c)
                new_msgs.append({"role": role, "content": new_content})
            else:
                new_msgs.append(m)
        valid_train_data.append({"messages": new_msgs})

print(f"🎯 Đã chuẩn bị {len(valid_train_data)} mẫu VQA hợp lệ có ảnh thật để huấn luyện!")


In [ ]:
# 4. Cấu hình Data Collator với Prompt Masking
class Qwen2VLCollator:
    def __init__(self, proc):
        self.processor = proc

    def __call__(self, batch):
        messages_list = [b["messages"] for b in batch]
        texts = [self.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages_list]
        image_inputs, video_inputs = process_vision_info(messages_list)
        inputs = self.processor(text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
        labels = inputs["input_ids"].clone()
        labels[inputs["attention_mask"] == 0] = -100
        
        im_start_id = self.processor.tokenizer.convert_tokens_to_ids("<|im_start|>")
        for i in range(inputs["input_ids"].size(0)):
            input_ids_list = inputs["input_ids"][i].tolist()
            assistant_start = -1
            for idx in range(len(input_ids_list) - 1, -1, -1):
                if input_ids_list[idx] == im_start_id:
                    cur = idx + 1
                    while cur < len(input_ids_list) and input_ids_list[cur] not in (198, 271) and cur < idx + 4:
                        cur += 1
                    while cur < len(input_ids_list) and input_ids_list[cur] in (198, 271):
                        cur += 1
                    assistant_start = cur
                    break
            if assistant_start != -1 and assistant_start < len(input_ids_list):
                labels[i, :assistant_start] = -100
            else:
                last_starts = [k for k, val in enumerate(input_ids_list) if val == im_start_id]
                if last_starts:
                    labels[i, :last_starts[-1] + 3] = -100
        inputs["labels"] = labels
        return inputs

collator = Qwen2VLCollator(processor)
print("✅ Khởi tạo thành công Qwen2VL Data Collator!")


In [ ]:
# 5. Nạp Base Model và Gắn QLoRA Config
print(f"⏳ Đang nạp Base Model: {model_id}...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("✅ Đã gắn thành công LoRA Adapter vào Qwen2-VL-2B!")


In [ ]:
# 6. Thiết lập Training Arguments và Huấn luyện
output_adapter_dir = "/kaggle/working/qwen2_vl_lora_adapters"
os.makedirs(output_adapter_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=3,
    max_steps=300,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=20,
    save_steps=100,
    save_total_limit=1,
    fp16=True,
    report_to="none",
    remove_unused_columns=False
)

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = SimpleDataset(valid_train_data)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator
)

print("=" * 75)
print("🚀 BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN LORA TRÊN TESLA T4...")
print("=" * 75)
trainer.train()

# Lưu trọng số LoRA Adapter
model.save_pretrained(output_adapter_dir)
processor.save_pretrained(output_adapter_dir)
print(f"💾 ĐÃ LƯU THÀNH CÔNG LORA ADAPTER TẠI: {output_adapter_dir}")


In [ ]:
# 7. Chạy Đánh Giá Đối Chứng Benchmark Sau Khi Fine-Tune
print("\n" + "=" * 75)
print("📊 [ĐÁNH GIÁ ĐỊNH LƯỢNG] CHẠY BENCHMARK TRÊN 15 LOẠI HÓA ĐƠN VỚI LORA MODEL...")
print("=" * 75)

def levenshtein_distance(s1: str, s2: str) -> int:
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def calculate_anls(prediction: str, ground_truth: str, threshold: float = 0.5) -> float:
    p = str(prediction).strip().lower()
    gt = str(ground_truth).strip().lower()
    if not p and not gt:
        return 1.0
    if not p or not gt:
        return 0.0
    dist = levenshtein_distance(p, gt)
    max_len = max(len(p), len(gt))
    norm_dist = dist / max_len
    if norm_dist < threshold:
        return round(1.0 - norm_dist, 4)
    return 0.0

def calculate_exact_match(prediction: str, ground_truth: str) -> float:
    return 1.0 if str(prediction).strip().lower() == str(ground_truth).strip().lower() else 0.0

def calculate_f1(prediction: str, ground_truth: str) -> float:
    pred_tokens = re.findall(r"\w+", str(prediction).lower())
    gt_tokens = re.findall(r"\w+", str(ground_truth).lower())
    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0
    common = set(pred_tokens) & set(gt_tokens)
    same_count = sum(min(pred_tokens.count(t), gt_tokens.count(t)) for t in common)
    if same_count == 0:
        return 0.0
    p = same_count / len(pred_tokens)
    r = same_count / len(gt_tokens)
    return round(2 * p * r / (p + r), 4)

model.eval()
# Nạp câu hỏi test thực tế từ 15 loại hóa đơn
eval_results = []
total_anls, total_em, total_f1 = 0.0, 0.0, 0.0
latencies = []
template_stats = {}

# Chạy suy luận trên các mẫu kiểm thử
# (Đọc trực tiếp từ multitemplate_validation_questions.json nếu có)
test_questions_file = "/kaggle/input/docvqa-benchmark-dataset/multitemplate_validation_questions.json"
test_items = []
if os.path.exists(test_questions_file):
    with open(test_questions_file, "r", encoding="utf-8") as f:
        test_items = json.load(f)[:45]

for idx, t_item in enumerate(test_items):
    img_name = t_item["image_name"]
    if img_name not in image_map:
        continue
    real_img = image_map[img_name]
    q = t_item["question"]
    gt = t_item["ground_truth"]
    tmpl = t_item.get("template", "unknown")
    
    t0 = time.time()
    im = Image.open(real_img).convert("RGB")
    msg = [{"role": "user", "content": [{"type": "image", "image": im}, {"type": "text", "text": q}]}]
    prompt_text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(msg)
    inps = processor(text=[prompt_text], images=imgs, videos=vids, padding=True, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        out_ids = model.generate(**inps, max_new_tokens=96, do_sample=False)
        trimmed = [o[len(i):] for i, o in zip(inps.input_ids, out_ids)]
        pred = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    
    lat = time.time() - t0
    latencies.append(lat)
    
    anls_v = calculate_anls(pred, gt)
    em_v = calculate_exact_match(pred, gt)
    f1_v = calculate_f1(pred, gt)
    
    total_anls += anls_v
    total_em += em_v
    total_f1 += f1_v
    
    if tmpl not in template_stats:
        template_stats[tmpl] = {"count": 0, "anls": 0.0, "em": 0.0, "f1": 0.0}
    template_stats[tmpl]["count"] += 1
    template_stats[tmpl]["anls"] += anls_v
    template_stats[tmpl]["em"] += em_v
    template_stats[tmpl]["f1"] += f1_v
    
    eval_results.append({
        "id": idx + 1,
        "template": tmpl,
        "image": img_name,
        "question": q,
        "ground_truth": gt,
        "prediction": pred,
        "anls": anls_v,
        "exact_match": int(em_v),
        "f1_score": f1_v,
        "latency_seconds": round(lat, 3)
    })

num_tests = len(eval_results)
avg_anls = total_anls / num_tests if num_tests > 0 else 0.0
avg_em = total_em / num_tests if num_tests > 0 else 0.0
avg_f1 = total_f1 / num_tests if num_tests > 0 else 0.0
avg_lat = sum(latencies) / len(latencies) if latencies else 0.0

template_breakdown = []
for t, d in template_stats.items():
    c = d["count"]
    template_breakdown.append({
        "template": t,
        "samples": c,
        "anls": f"{d['anls']/c*100:.2f}%" if c > 0 else "0%",
        "exact_match": f"{d['em']/c*100:.2f}%" if c > 0 else "0%",
        "f1_score": f"{d['f1']/c*100:.2f}%" if c > 0 else "0%"
    })

lora_report = {
    "model_name": "Qwen2-VL-2B + QLoRA (Rank 16, Alpha 32 - Fine-Tuned)",
    "hardware": f"Kaggle GPU {torch.cuda.get_device_name(0)}",
    "total_test_records": num_tests,
    "anls_score": round(avg_anls, 4),
    "anls_percentage": f"{avg_anls * 100:.2f}%",
    "exact_match_rate": round(avg_em, 4),
    "exact_match_percentage": f"{avg_em * 100:.2f}%",
    "f1_score": round(avg_f1, 4),
    "f1_percentage": f"{avg_f1 * 100:.2f}%",
    "avg_latency_seconds": round(avg_lat, 3),
    "vram_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024**3), 2),
    "adapter_size_mb": 73.9,
    "template_breakdown": template_breakdown,
    "details": eval_results
}

with open("/kaggle/working/evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(lora_report, f, ensure_ascii=False, indent=2)

# Nén thư mục adapter để tải về
!cd /kaggle/working && zip -r qwen2_vl_lora_adapters.zip qwen2_vl_lora_adapters

print("\n" + "=" * 75)
print("🏆 TỔNG HỢP HIỆU NĂNG LORA MODEL SAU KHI HUẤN LUYỆN:")
print("=" * 75)
print(f"- ANLS Score      : {lora_report['anls_score']} ({lora_report['anls_percentage']})")
print(f"- Exact Match (EM): {lora_report['exact_match_rate']} ({lora_report['exact_match_percentage']})")
print(f"- F1-Score        : {lora_report['f1_score']} ({lora_report['f1_percentage']})")
print(f"- Latency GPU T4  : {lora_report['avg_latency_seconds']}s / câu hỏi")
print("=" * 75)
